In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *
from rapidfuzz import process
from rapidfuzz import fuzz

In [5]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/V3/rows_output/'
os.makedirs(output_folder, exist_ok=True)


In [16]:
# === 1. Read input data ===
file_path = os.path.join(output_folder, "Fern_raw_db_station_matched_v1.csv")
match_1_pd = pd.read_csv(file_path)
match_1_pd[['variable','db_var_match']]

,variable,db_var_match
0,Rain,NaN
1,Air Temp,NaN
2,RH,NaN
3,DewPt,NaN
4,Wind Speed,NaN
...,...,...
428,Wind Speed,WindSpeedms
429,Gust Speed,GustSpeedms
430,Wind Direction,WindDirection
431,Solar Radiation,SolarRadiationWm


In [20]:
# Keep only rows where db_var_match is not NaN
mapping_df = (
    match_1_pd[['variable', 'db_var_match']]
    .dropna(subset=['db_var_match'])      # remove NaN mappings
    .drop_duplicates(subset=['variable']) # remove duplicates by variable
    .reset_index(drop=True)
)

print(mapping_df)

           variable      db_var_match
0              Rain            Rainmm
1          Pressure      Pressurembar
2              Temp             TempC
3                RH                RH
4             DewPt            DewPtC
5        Wind Speed       WindSpeedms
6        Gust Speed       GustSpeedms
7    Wind Direction     WindDirection
8   Solar Radiation  SolarRadiationWm
9         Soil Temp         Soil_Temp
10       Snow depth        Snow_Depth
11          Wetness           Wetness


In [79]:
# === 1. Read input data ===
file_path = os.path.join(output_folder, "Fern_raw_db_station_matched_v1.csv")
match_1_pd = pd.read_csv(file_path)

# === 2. Mark whether each variable has already been inserted ===
match_1_pd["Has_inserted"] = match_1_pd["db_var_match"].apply(
    lambda x: "No" if pd.isna(x) else "Yes"
)

# === 3. Select only rows that have not been inserted ===
match_2_pd = (
    match_1_pd.loc[match_1_pd["Has_inserted"] == "No"]
    .drop(columns=["variable_match_score"], errors="ignore")
    .drop_duplicates()
)

# === 4. Prepare variables for fuzzy matching ===
var_db = match_1_pd["db_var_match"].dropna().drop_duplicates()
threshold = 80
results = []

# === 5. Perform fuzzy matching ===
for file_var in match_2_pd["variable"]:
    match_result = process.extractOne(file_var, var_db, scorer=fuzz.ratio)
    if match_result:
        match, score, _ = match_result
    else:
        match, score = None, 0

    results.append({
        "variable": file_var,
        "db_var_match": match if score >= threshold else None,
        "score": score,
    })

# === 6. Merge results back into match_2_pd ===
results_df = pd.DataFrame(results)

match_2_pd = (
    match_2_pd
    .drop(columns=["db_var_match", "score"], errors="ignore")
    .merge(results_df, on="variable", how="left")
    .drop_duplicates()
)

# === 7. Fill in unit_db using match_1_pd lookup ===
unit_lookup = (
    match_1_pd[["db_var_match", "unit_db"]]
    .dropna(subset=["db_var_match"])
    .drop_duplicates()
)

match_2_pd = (
    match_2_pd
    .drop(columns=["unit_db"], errors="ignore")
    .merge(unit_lookup, on="db_var_match", how="left")
)

# === 8. Reorder columns ===
cols = list(match_2_pd.columns)
# Move db_var_match right after latest_time_raw
if "db_var_match" in cols and "latest_time_raw" in cols:
    cols.insert(cols.index("latest_time_raw") + 1, cols.pop(cols.index("db_var_match")))
# Move unit_db right after db_var_match
if "db_var_match" in cols and "unit_db" in cols:
    cols.insert(cols.index("db_var_match") + 1, cols.pop(cols.index("unit_db")))
match_2_pd = match_2_pd[cols]

# === 9. Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_matched_v2_raw_db_station.csv")
match_2_pd.to_csv(output_file, index=False)

# === 10. Summary ===
print(f"✅ Matching complete! Saved to: {output_file}")
print(f"Matched {len(results_df)} variables (threshold = {threshold}).")
print(f"Remaining unmatched stations: {match_2_pd['station_name'].nunique()}")

✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_matched_v2_raw_db_station.csv
Matched 155 variables (threshold = 80).
Remaining unmatched stations: 29


#### First deal with the 6 new stations which are not in the database

In [ ]:
# === 1. Read input data ===
file_path = os.path.join(output_folder, "station_location_compare.csv")
station_match = pd.read_csv(file_path)

not_match_station = station_match[pd.isna(station_match['db_matched_name'])]
not_match_station_name =not_match_station['station_name']
not_match_station_inset_batch2 = match_2_pd[match_2_pd["station_name"].isin(not_match_station_name)]

filtered = not_match_station_inset_batch2[not_match_station_inset_batch2["db_var_match"].notna()].reset_index(drop=True)

In [73]:
from row_generate_func import *

data_path = '/fern_data/FERNNorth2024_VF/WxData24'

all_rows = []

for i, row in filtered.iterrows():
    station = row["station_name"]
    variable = row["variable"]

    print(f"\n🔍 [{i+1}/{len(filtered)}] Processing {station} — {variable}")

    new_rows = extract_new_data_rows(row, data_path)
    all_rows.extend(new_rows)

    if len(new_rows) > 0:
        print(f"  ✅ {len(new_rows)} new records found outside DB time range.")
    else:
        print("  ⚠️ No new records found (matches DB time range exactly).")


🔍 [1/58] Processing Atlin School — Rain
  ✅ 119231 new records found outside DB time range.

🔍 [2/58] Processing Atlin School — RH
  ✅ 111039 new records found outside DB time range.

🔍 [3/58] Processing Atlin School — DewPt
  ✅ 111039 new records found outside DB time range.

🔍 [4/58] Processing Atlin School — Wind Speed
  ✅ 119237 new records found outside DB time range.

🔍 [5/58] Processing Atlin School — Gust Speed
  ✅ 119237 new records found outside DB time range.

🔍 [6/58] Processing Atlin School — Wind Direction
  ✅ 119237 new records found outside DB time range.

🔍 [7/58] Processing Atlin School — Solar Radiation
  ✅ 119237 new records found outside DB time range.

🔍 [8/58] Processing CrookedLk — Rain
  ✅ 9572 new records found outside DB time range.

🔍 [9/58] Processing CrookedLk — Pressure
  ✅ 9286 new records found outside DB time range.

🔍 [10/58] Processing CrookedLk — Temp
  ✅ 9536 new records found outside DB time range.

🔍 [11/58] Processing CrookedLk — RH
  ✅ 9558 ne

In [ ]:
# all_rows

# out_file = os.path.join(output_folder, f'new_6_stations_rows.pickle')
# with open(out_file, 'wb') as f:
#     pickle.dump(all_rows, f)


### Insert

In [75]:
from crmprtd.infer import infer

rows_insert = all_rows

for i in range(0, len(rows_insert), 10000):  # test in chunks of 100
    chunk = rows_insert[i:i+100]
    try:
        infer(session, chunk)
        print(f"✅ Rows {i}–{i+len(chunk)-1} OK")
    except ValueError as e:
        print(f"❌ Rows {i}–{i+len(chunk)-1} failed: {e}")
        break  # stop and debug this smaller chunk

✅ Rows 0–99 OK
✅ Rows 10000–10099 OK
✅ Rows 20000–20099 OK
✅ Rows 30000–30099 OK
✅ Rows 40000–40099 OK
✅ Rows 50000–50099 OK
✅ Rows 60000–60099 OK
✅ Rows 70000–70099 OK
✅ Rows 80000–80099 OK
✅ Rows 90000–90099 OK
✅ Rows 100000–100099 OK
✅ Rows 110000–110099 OK
✅ Rows 120000–120099 OK
✅ Rows 130000–130099 OK
✅ Rows 140000–140099 OK
✅ Rows 150000–150099 OK
✅ Rows 160000–160099 OK
✅ Rows 170000–170099 OK
✅ Rows 180000–180099 OK
✅ Rows 190000–190099 OK
✅ Rows 200000–200099 OK
✅ Rows 210000–210099 OK
✅ Rows 220000–220099 OK
✅ Rows 230000–230099 OK
✅ Rows 240000–240099 OK
✅ Rows 250000–250099 OK
✅ Rows 260000–260099 OK
✅ Rows 270000–270099 OK
✅ Rows 280000–280099 OK
✅ Rows 290000–290099 OK
✅ Rows 300000–300099 OK
✅ Rows 310000–310099 OK
✅ Rows 320000–320099 OK
✅ Rows 330000–330099 OK
✅ Rows 340000–340099 OK
✅ Rows 350000–350099 OK
✅ Rows 360000–360099 OK
✅ Rows 370000–370099 OK
✅ Rows 380000–380099 OK
✅ Rows 390000–390099 OK
✅ Rows 400000–400099 OK
✅ Rows 410000–410099 OK
✅ Rows 420000–42009

In [76]:
### Auto insert failed
from auto_insert_func import *

rows_insert = all_rows
db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = '/workspaces/crmprtd/fern/all_stations_insert/fern_all_station_log.log'
insert_rows(rows_insert, log_file_path, db_url)


{"asctime": "2025-11-12 19:55:29,567", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = CrookedLk"}
{"asctime": "2025-11-12 19:55:29,655", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-12 19:55:29,684", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: find_or_create_matching_history_and_station ('FLNRO-FERN', 'CrookedLk', 59.88626, -126.42886, False) -> <pycds.orm.tables.History object at 0xffff03fa2650>"}
{"asctime": "2025-11-12 19:55:30,155", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = IBB2"}
{"asctime": "2025-11-12 19:55:30,160", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-12 19:55:30,162", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: find_or_create_matching_history_and_station ('FLNRO-FERN', 'IBB2', 54.81127, -1

{'successes': 1146002,
 'skips': 50007,
 'failures': 1000,
 'insertions_per_sec': 1476.18}

### After this step, update the insert condition of these 5 staions as "Yes"

In [ ]:
file_path = os.path.join(output_folder, "Fern_matched_v2_raw_db_station.csv")
match_2_pd = pd.read_csv(file_path)


file_path = os.path.join(output_folder, "station_location_compare.csv")
station_match = pd.read_csv(file_path)

not_match_station = station_match[pd.isna(station_match['db_matched_name'])]
not_match_station_name =not_match_station['station_name']
not_match_station_inset_batch2 = match_2_pd[match_2_pd["station_name"].isin(not_match_station_name)]

filtered = not_match_station_inset_batch2[not_match_station_inset_batch2["db_var_match"].notna()]

# Update 'Has_inserted' directly in the original DataFrame using the subset's index
match_2_pd.loc[filtered.index, 'Has_inserted'] = 'Yes'

## Update this into the csv.files
# === Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_matched_v2_raw_db_station.csv")
match_2_pd.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")


✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_matched_v2_raw_db_station.csv


In [ ]:
subset_no = match_2_pd[match_2_pd["Has_inserted"] == "No"].copy()
subset_no

# === Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_manually_matched_v3_raw_db_station.csv")
subset_no.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")


✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_manually_matched_v3_raw_db_station.csv
